In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Transpiler Service를 사용하여 원격으로 Circuit 트랜스파일하기

> **Danger:** 2025년 7월 18일부터 해당 서비스는 새로운 IBM Quantum&reg; Platform을 지원하기 위해 마이그레이션 중이며 현재 사용할 수 없습니다. AI 패스의 경우 [로컬 모드](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud)를 사용할 수 있습니다.
> 
>     이 서비스는 베타 버전으로, 변경될 수 있습니다.
>     피드백이 있거나 개발팀에 문의하고 싶으시다면 이 [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU) 채널을 이용해 주세요.

Qiskit Transpiler Service는 클라우드에서 트랜스파일 기능을 제공합니다. 로컬 Qiskit Transpiler 기능 외에도, 트랜스파일 작업은 IBM Quantum 클라우드 리소스와 AI 기반 Transpiler 패스의 혜택을 받을 수 있습니다.

Qiskit Transpiler Service는 이 서비스와 그 기능을 현재의 Qiskit 패턴 및 워크플로에 원활하게 통합할 수 있는 Python 라이브러리를 제공합니다. 이 서비스는 IBM Quantum Premium Plan, Flex Plan, On-Prem(IBM Quantum Platform API를 통한) Plan 사용자만 이용할 수 있습니다.

<span id="install-transpiler-service"></span>
## qiskit-ibm-transpiler 패키지 설치
Qiskit Transpiler Service를 사용하려면 `qiskit-ibm-transpiler` 패키지를 설치하세요:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

이 패키지는 [Qiskit Runtime이 관리하는 방식](/guides/initialize-account)에 맞게 [IBM Quantum Platform 자격 증명](/guides/cloud-setup)을 사용하여 자동으로 인증합니다:
- 환경 변수: `QISKIT_IBM_TOKEN`
- 구성 파일 `~/.qiskit/qiskit-ibm.json` (`default-ibm-quantum` 섹션 아래).

*참고*: 이 패키지는 Qiskit SDK v1.X가 필요합니다.

## qiskit-ibm-transpiler 트랜스파일 옵션
- `backend_name` (선택 사항, str) - QiskitRuntimeService에서 예상하는 Backend 이름(예: `ibm_torino`). 이 값이 설정되면, 트랜스파일 메서드는 트랜스파일 작업에 지정된 Backend의 레이아웃을 사용합니다. `coupling_map`과 같이 이 설정에 영향을 미치는 다른 옵션이 설정되어 있으면 `backend_name` 설정이 재정의됩니다.
- `coupling_map` (선택 사항, List[List[int]]) - 유효한 결합 맵 목록(예: [[0,1],[1,2]]). 이 값이 설정되면, 트랜스파일 메서드는 트랜스파일 작업에 이 결합 맵을 사용합니다. 정의된 경우, `target`에 지정된 값을 재정의합니다.
- `optimization_level` (int) - 트랜스파일 과정에서 적용할 잠재적 최적화 수준. 유효한 값은 [1,2,3]이며, 1은 최소 최적화(가장 빠름), 3은 최대 최적화(가장 시간이 많이 걸림)를 의미합니다.
- `ai` ("true", "false", "auto") - 트랜스파일 중 AI 기반 기능을 사용할지 여부. 사용 가능한 AI 기반 기능은 `AIRouting` 트랜스파일 패스 또는 기타 AI 기반 합성 방법일 수 있습니다. 이 값이 `"true"`이면, 서비스는 요청된 `optimization_level`에 따라 다양한 AI 기반 트랜스파일 패스를 적용합니다. `"false"`이면, AI 없이 최신 Qiskit 트랜스파일 기능을 사용합니다. 마지막으로 `"auto"`이면, 서비스가 Circuit에 따라 표준 Qiskit 휴리스틱 패스 또는 AI 기반 패스를 적용할지 결정합니다.
- `qiskit_transpile_options` (dict) - [Qiskit `transpile()` 메서드](defaults-and-configuration-options)에서 유효한 다른 옵션을 포함할 수 있는 Python 딕셔너리 객체. `qiskit_transpile_options` 입력에 `optimization_level`이 포함되면, 파라미터 입력으로 지정된 `optimization_level`을 우선시하여 무시됩니다. `qiskit_transpile_options`에 Qiskit `transpile()` 메서드가 인식하지 못하는 옵션이 포함되면 라이브러리에서 오류가 발생합니다.

사용 가능한 `qiskit-ibm-transpiler` 메서드에 대한 자세한 내용은 [qiskit-ibm-transpiler API 참조](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler)를 참조하세요. 서비스 API에 대해 자세히 알아보려면 [Qiskit Transpiler Service REST API 문서](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)를 참조하세요.

## 예제
다음 예제는 다양한 파라미터를 사용하여 Qiskit Transpiler Service로 Circuit을 트랜스파일하는 방법을 보여줍니다.

1. Circuit을 생성하고 Qiskit Transpiler Service를 호출하여 `backend_name`으로 `ibm_torino`, `optimization_level`로 3을 설정하고 트랜스파일 중 AI를 사용하지 않고 Circuit을 트랜스파일합니다.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*참고*: IBM Quantum 계정으로 접근 권한이 있는 backend_name 디바이스만 사용할 수 있습니다. `backend_name` 외에도 `TranspilerService`는 `coupling_map`을 파라미터로 허용합니다.

2. 유사한 Circuit을 생성하고 `ai` 플래그를 `True`로 설정하여 AI 트랜스파일 기능을 요청하며 트랜스파일합니다:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. 유사한 Circuit을 생성하고 서비스가 AI 기반 트랜스파일 패스를 사용할지 여부를 결정하도록 하면서 트랜스파일합니다.